1. Prepare PySpark

In [ ]:
# Installing required packages
!pip install pyspark
!pip install findspark

In [ ]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

In [ ]:
from pyspark.sql import SQLContext

In [ ]:
# Create a Spark session
spark = SparkSession.builder \
    .appName("SpotifyRegression") \
    .getOrCreate()


In [ ]:
sc = spark.sparkContext
sqlContext = SQLContext(sc)

/usr/local/lib/python3.11/dist-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


2.Prepare Feature

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving spotify-data.csv to spotify-data.csv


In [ ]:
import pandas as pd
file = 'spotify-data.csv'
df = pd.read_csv(file)
df.head()

,id,name,artists,duration_ms,release_date,year,acousticness,danceability,energy,instrumentalness,liveness,loudness,speechiness,tempo,valence,mode,key,popularity,explicit
0,6KbQ3uYMLKb5jDxLF7wYDD,Singende Bataillone 1. Teil,['Carl Woitschach'],158648,1928,1928,0.995,0.708,0.1950,0.563,0.1510,-12.428,0.0506,118.469,0.7790,1,10,0,0
1,6KuQTIu1KoTTkLXKrwlLPV,"Fantasiestücke, Op. 111: Più tosto lento","['Robert Schumann', 'Vladimir Horowitz']",282133,1928,1928,0.994,0.379,0.0135,0.901,0.0763,-28.454,0.0462,83.972,0.0767,1,8,0,0
2,6L63VW0PibdM1HDSBoqnoM,Chapter 1.18 - Zamek kaniowski,['Seweryn Goszczyński'],104300,1928,1928,0.604,0.749,0.2200,0.000,0.1190,-19.924,0.9290,107.177,0.8800,0,5,0,0
3,6M94FkXd15sOAOQYRnWPN8,Bebamos Juntos - Instrumental (Remasterizado),['Francisco Canaro'],180760,9/25/28,1928,0.995,0.781,0.1300,0.887,0.1110,-14.734,0.0926,108.003,0.7200,0,1,0,0
4,6N6tiFZ9vLTSOIxkj8qKrd,"Polonaise-Fantaisie in A-Flat Major, Op. 61","['Frédéric Chopin', 'Vladimir Horowitz']",687733,1928,1928,0.990,0.210,0.2040,0.908,0.0980,-16.829,0.0424,62.149,0.0693,1,11,1,0


In [ ]:
df = spark.createDataFrame(df)

In [ ]:
df.columns

['id',
 'name',
 'artists',
 'duration_ms',
 'release_date',
 'year',
 'acousticness',
 'danceability',
 'energy',
 'instrumentalness',
 'liveness',
 'loudness',
 'speechiness',
 'tempo',
 'valence',
 'mode',
 'key',
 'popularity',
 'explicit']

In [ ]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- duration_ms: long (nullable = true)
 |-- release_date: string (nullable = true)
 |-- year: long (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- mode: long (nullable = true)
 |-- key: long (nullable = true)
 |-- popularity: long (nullable = true)
 |-- explicit: long (nullable = true)



In [ ]:
df.describe().toPandas().transpose()

,0,1,2,3,4
summary,count,mean,stddev,min,max
id,169909,None,None,000G1xMMuwxNHmwVsBdtj1,7zzuPsjj9L3M7ikqGmjN0D
name,169909,Infinity,NaN,!Que Vida! - Mono Version,화려하지 않은 고백 Confession Is Not Flashy
artists,169909,None,None,"[""'In The Heights' Original Broadway Company"",...",['黑豹']
duration_ms,169909,231406.1589733328,121321.92321940252,5108,5403500
release_date,169909,1963.7319876146244,21.57082683232273,1/1/00,9/9/98
year,169909,1977.2232312590857,25.59316763176324,1921,2020
acousticness,169909,0.49321397614987705,0.3766270623378325,0.0,0.996
danceability,169909,0.5381497172015588,0.17534578204760842,0.0,0.988
energy,169909,0.4885931303603691,0.2673899329571335,0.0,1.0


In [ ]:
df.describe('danceability').show()

+-------+-------------------+
|summary|       danceability|
+-------+-------------------+
|  count|             169909|
|   mean| 0.5381497172015588|
| stddev|0.17534578204760842|
|    min|                0.0|
|    max|              0.988|
+-------+-------------------+



Regression

In [ ]:
from pyspark.sql import DataFrameNaFunctions
from pyspark.ml.feature import VectorAssembler

In [ ]:
df = df.na.drop() #Dropping all rows with missing values

In [ ]:
df = df.withColumnRenamed("danceability", "label")

In [ ]:
# Select only numeric features
from pyspark.sql.types import StringType

featureColumns = [field.name for field in df.schema.fields
                  if not isinstance(field.dataType, StringType) and field.name != 'label']

In [ ]:
featureColumns

['duration_ms',
 'year',
 'acousticness',
 'energy',
 'instrumentalness',
 'liveness',
 'loudness',
 'speechiness',
 'tempo',
 'valence',
 'mode',
 'key',
 'popularity',
 'explicit']

In [ ]:
# Assemble features into vector
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=featureColumns, outputCol="features") # outputCol has a default name: features.
assembled = assembler.transform(df)

In [ ]:
assembled.show(10)

+--------------------+--------------------+--------------------+-----------+------------+----+------------+-----+------+----------------+--------+--------+-----------+-------+-------+----+---+----------+--------+--------------------+
|                  id|                name|             artists|duration_ms|release_date|year|acousticness|label|energy|instrumentalness|liveness|loudness|speechiness|  tempo|valence|mode|key|popularity|explicit|            features|
+--------------------+--------------------+--------------------+-----------+------------+----+------------+-----+------+----------------+--------+--------+-----------+-------+-------+----+---+----------+--------+--------------------+
|6KbQ3uYMLKb5jDxLF...|Singende Bataillo...| ['Carl Woitschach']|     158648|        1928|1928|       0.995|0.708| 0.195|           0.563|   0.151| -12.428|     0.0506|118.469|  0.779|   1| 10|         0|       0|[158648.0,1928.0,...|
|6KuQTIu1KoTTkLXKr...|Fantasiestücke, O...|['Robert Schumann...|

In [ ]:
select_assembled = assembled.select("label", "features")

In [ ]:
(trainData, testData) = select_assembled.randomSplit([0.8,0.2], seed = 13234 )

In [ ]:
trainData.count(),testData.count()

(135820, 34089)

In [ ]:
from pyspark.ml.regression import LinearRegression

In [ ]:
trainData.show()

+-----+--------------------+
|label|            features|
+-----+--------------------+
|  0.0|(14,[0,1,2,3,4,5,...|
|  0.0|(14,[0,1,2,3,5,6,...|
|  0.0|(14,[0,1,2,3,5,6,...|
|  0.0|(14,[0,1,2,3,5,6,...|
|  0.0|(14,[0,1,2,3,5,6,...|
|  0.0|(14,[0,1,3,4,5,6,...|
|  0.0|(14,[0,1,3,4,5,6,...|
|  0.0|(14,[0,1,3,5,6,11...|
|  0.0|(14,[0,1,6],[6467...|
|  0.0|(14,[0,1,6,12],[1...|
|  0.0|(14,[0,1,6,12],[1...|
|  0.0|(14,[0,1,6,12],[2...|
|  0.0|[8853.0,1954.0,0....|
|  0.0|[14708.0,1973.0,0...|
|  0.0|[14960.0,1952.0,0...|
|  0.0|[42107.0,1955.0,0...|
|  0.0|[57467.0,1977.0,0...|
|  0.0|[72652.0,2014.0,0...|
|  0.0|[93452.0,2014.0,0...|
|  0.0|[93452.0,2015.0,0...|
+-----+--------------------+
only showing top 20 rows



In [ ]:
testData.show()

+------+--------------------+
| label|            features|
+------+--------------------+
|   0.0|(14,[0,1,6,12],[6...|
|   0.0|[55693.0,1964.0,0...|
|   0.0|[60280.0,1966.0,0...|
|   0.0|[75000.0,2018.0,0...|
|   0.0|[93452.0,2016.0,0...|
|   0.0|[104516.0,2017.0,...|
|   0.0|[158984.0,2017.0,...|
|   0.0|[159600.0,1937.0,...|
|   0.0|[170614.0,2017.0,...|
|   0.0|[180656.0,2017.0,...|
|   0.0|[192130.0,1948.0,...|
|0.0583|[600200.0,1962.0,...|
|0.0587|[272533.0,1954.0,...|
|0.0596|[484013.0,2009.0,...|
|0.0603|[304655.0,2009.0,...|
|0.0604|[219693.0,2005.0,...|
|0.0604|[232871.0,2013.0,...|
|0.0608|[239547.0,1948.0,...|
|0.0613|[471368.0,1935.0,...|
|0.0614|[416387.0,2000.0,...|
+------+--------------------+
only showing top 20 rows



In [ ]:
# Train Linear Regression
lr = LinearRegression()

In [ ]:
model1 = lr.fit(trainData)

In [ ]:
print("Coefficients: %s" % str(model1.coefficients))
print("Intercept: %s" % str(model1.intercept))

Coefficients: [-1.3899758275798344e-08,0.0009431786395430103,-0.06354384898618852,-0.23360102672282992,-0.02675260428862082,-0.09198184653371291,0.0067257076004748825,0.20849135092309923,-0.00071313258867999,0.40094846102596315,-0.012181908690179162,-8.986536871792839e-05,0.0006969729248900226,0.06741590081218546]
Intercept: -1.2462776949413663


In [ ]:
# Evaluate
trainingSummary = model1.summary
print("numIterations: %d" % trainingSummary.totalIterations)
print("objectiveHistory: %s" % str(trainingSummary.objectiveHistory))
trainingSummary.residuals.show()
print("RMSE: %f" % trainingSummary.rootMeanSquaredError)
print("r2: %f" % trainingSummary.r2)

numIterations: 0
objectiveHistory: [0.0]
+--------------------+
|           residuals|
+--------------------+
|-0.38018878528707756|
| -0.2649052313093643|
|-0.29862062594004213|
| -0.2799422286723072|
| -0.2904053836892313|
| -0.5005288537132908|
|  -0.504259924052394|
| -0.4812279302228599|
|-0.18834512776269818|
|  -0.269172032396064|
|-0.20467912361439655|
|-0.20364702486314368|
| -0.2263523317025007|
|-0.42506746254903316|
| -0.2830733329847579|
|-0.19692615665691493|
| -0.4585419003787039|
| -0.5383786007484685|
|  -0.449410800163186|
| -0.4857670216194563|
+--------------------+
only showing top 20 rows

RMSE: 0.124415
r2: 0.496841


In [ ]:
predictions = model1.transform(testData)

In [ ]:
predictions.show()

+------+--------------------+-------------------+
| label|            features|         prediction|
+------+--------------------+-------------------+
|   0.0|(14,[0,1,6,12],[6...| 0.2582056670015993|
|   0.0|[55693.0,1964.0,0...| 0.2853874758226742|
|   0.0|[60280.0,1966.0,0...| 0.3167233014686275|
|   0.0|[75000.0,2018.0,0...| 0.5078729726856495|
|   0.0|[93452.0,2016.0,0...|  0.453388076216942|
|   0.0|[104516.0,2017.0,...| 0.4124568507380566|
|   0.0|[158984.0,2017.0,...| 0.1515473686414055|
|   0.0|[159600.0,1937.0,...| 0.3429407257289583|
|   0.0|[170614.0,2017.0,...| 0.5236080117855371|
|   0.0|[180656.0,2017.0,...| 0.4626693441758474|
|   0.0|[192130.0,1948.0,...| 0.4055102042342549|
|0.0583|[600200.0,1962.0,...| 0.3132146060079113|
|0.0587|[272533.0,1954.0,...|0.26136160055245994|
|0.0596|[484013.0,2009.0,...|  0.364717741427552|
|0.0603|[304655.0,2009.0,...| 0.3602958294826839|
|0.0604|[219693.0,2005.0,...| 0.3874257596702493|
|0.0604|[232871.0,2013.0,...|0.40481732412899807|


In [ ]:
predictions.select("label", "prediction").show()


+------+-------------------+
| label|         prediction|
+------+-------------------+
|   0.0| 0.2582056670015993|
|   0.0| 0.2853874758226742|
|   0.0| 0.3167233014686275|
|   0.0| 0.5078729726856495|
|   0.0|  0.453388076216942|
|   0.0| 0.4124568507380566|
|   0.0| 0.1515473686414055|
|   0.0| 0.3429407257289583|
|   0.0| 0.5236080117855371|
|   0.0| 0.4626693441758474|
|   0.0| 0.4055102042342549|
|0.0583| 0.3132146060079113|
|0.0587|0.26136160055245994|
|0.0596|  0.364717741427552|
|0.0603| 0.3602958294826839|
|0.0604| 0.3874257596702493|
|0.0604|0.40481732412899807|
|0.0608| 0.2667799708221201|
|0.0613|0.27409921842614327|
|0.0614|0.35601767859171174|
+------+-------------------+
only showing top 20 rows



#Predict others things

In [ ]:
featureColumns1 = df.columns[5:]
featureColumns1

['year',
 'acousticness',
 'label',
 'energy',
 'instrumentalness',
 'liveness',
 'loudness',
 'speechiness',
 'tempo',
 'valence',
 'mode',
 'key',
 'popularity',
 'explicit']